# E-Commerce PySpark Structured Streaming Consumer

Reads enriched order-item events from a Kafka topic and applies a multi-stage
transformation pipeline before persisting results to MongoDB via `foreachBatch`.

## Pipeline stages
1. **Ingest** – read JSON messages from Kafka
2. **Parse** – deserialize JSON payload, enforce schema, cast types
3. **Enrich** – derive computed columns (`line_total`, `event_timestamp`)
4. **Aggregation A** – windowed revenue + order count per product category (1-day tumbling window)
5. **Aggregation B** – order count per shipping country
6. **Aggregation C** – payment method distribution per window
7. **Sink** – write each micro-batch to MongoDB via `foreachBatch`

## MongoDB collections
| Collection | Description |
|---|---|
| `revenue_by_category` | Windowed category aggregates |
| `orders_by_country` | Windowed country aggregates |
| `payment_method_stats` | Windowed payment method counts |
| `orders_raw` | Enriched raw events |

## How to run the producer
```bash
docker exec -it spark-notebook /bin/bash
python3 /opt/spark/work-dir/src/producers/producer.py \
    --broker kafka:9093 \
    --topic ecommerce-orders \
    --records 5000 \
    --delay 0.2
```

## 1. Create SparkSession

In [ ]:
from SparkUtils import SparkUtils

from pathlib import Path
import shutil
import pyspark.sql.functions as F
from pyspark.sql import DataFrame

packages = ",".join([
    "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0",
    "org.mongodb.spark:mongo-spark-connector_2.13:10.5.0",
])

su = SparkUtils(
    "E-Commerce Stream Processing with Kafka",
    master_url="local[*]",
    packages=packages,
)
su.spark.conf.set("spark.mongodb.write.connection.uri", "mongodb://mongo:27017")
su.spark

## 2. Configuration

In [ ]:
KAFKA_SERVERS  = "kafka:9093"
KAFKA_TOPIC    = "ecommerce-orders"
MONGO_URI      = "mongodb://mongo:27017"
MONGO_DATABASE = "ecommerce"
CHECKPOINT_DIR = "/opt/spark/work-dir/notebooks/checkpoints"

# Stop any existing streaming queries before starting new ones
for q in su.spark.streams.active:
    q.stop()

# Clean up old checkpoints so the stream starts fresh
checkpoint_path = Path(CHECKPOINT_DIR)
if checkpoint_path.exists() and checkpoint_path.is_dir():
    shutil.rmtree(checkpoint_path)
    print(f"Cleared checkpoint directory: {CHECKPOINT_DIR}")

print(f"Kafka  : {KAFKA_SERVERS}  →  topic: {KAFKA_TOPIC}")
print(f"MongoDB: {MONGO_URI}  →  db: {MONGO_DATABASE}")

## 3. Schema definition

In [ ]:
ORDER_ITEM_SCHEMA = SparkUtils.generate_schema([
    ("order_item_id",    "string"),
    ("order_id",         "string"),
    ("product_id",       "string"),
    ("quantity",         "int"),
    ("unit_price",       "double"),
    ("customer_id",      "string"),
    ("order_date",       "string"),
    ("total_amount",     "double"),
    ("payment_method",   "string"),
    ("shipping_country", "string"),
    ("product_name",     "string"),
    ("category",         "string"),
    ("product_price",    "double"),
    ("brand",            "string"),
    ("customer_name",    "string"),
    ("country",          "string"),
])

print(f"Schema has {len(ORDER_ITEM_SCHEMA.fields)} fields")

## 4. Stage 1 & 2 — Ingest + Parse

In [ ]:
raw_stream = (
    su.spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_SERVERS)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .option("failOnDataLoss", "false")
    .load()
    .select(F.from_json(F.col("value").cast("string"), ORDER_ITEM_SCHEMA).alias("data"))
    .select("data.*")
)

print("Kafka stream schema:")
raw_stream.printSchema()

## 5. Stage 3 — Enrich

In [ ]:
enriched = (
    raw_stream
    .withColumn("line_total", F.col("quantity") * F.col("unit_price"))
    .withColumn("event_timestamp", F.try_to_timestamp(F.col("order_date"), F.lit("yyyy-MM-dd")))
    .withColumn("processing_time", F.current_timestamp())
    .filter(F.col("event_timestamp").isNotNull())
    .filter(F.col("line_total") > 0)
)

print("Enriched stream schema:")
enriched.printSchema()

## 6. Stage 4 — Aggregation A: Revenue by Category

In [ ]:
cat_stream = (
    enriched
    .groupBy(
        F.window("event_timestamp", "1 day").alias("window"),
        F.col("category"),
    )
    .agg(
        F.sum("line_total").alias("total_revenue"),
        F.approx_count_distinct("order_id").alias("total_orders"),
        F.avg("line_total").alias("avg_line_value"),
        F.sum("quantity").alias("total_units_sold"),
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "category",
        F.round("total_revenue", 2).alias("total_revenue"),
        "total_orders",
        F.round("avg_line_value", 2).alias("avg_line_value"),
        "total_units_sold",
    )
)

print("revenue_by_category schema:")
cat_stream.printSchema()

## 7. Stage 5 — Aggregation B: Orders by Country

In [ ]:
country_stream = (
    enriched
    .groupBy(
        F.window("event_timestamp", "1 day").alias("window"),
        F.col("shipping_country"),
    )
    .agg(
        F.approx_count_distinct("order_id").alias("order_count"),
        F.sum("line_total").alias("total_revenue"),
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "shipping_country",
        "order_count",
        F.round("total_revenue", 2).alias("total_revenue"),
    )
)

print("orders_by_country schema:")
country_stream.printSchema()

## 8. Stage 6 — Aggregation C: Payment Method Stats

In [ ]:
payment_stream = (
    enriched
    .groupBy(
        F.window("event_timestamp", "1 day").alias("window"),
        F.col("payment_method"),
    )
    .agg(
        F.count("order_item_id").alias("transaction_count"),
        F.sum("total_amount").alias("total_amount"),
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "payment_method",
        "transaction_count",
        F.round("total_amount", 2).alias("total_amount"),
    )
)

print("payment_method_stats schema:")
payment_stream.printSchema()

## 9. Stage 7 — MongoDB Sink via `foreachBatch`

Aggregation collections use `mode("overwrite")` — since `complete` output already recomputes the full result each batch, overwriting the collection is correct and avoids duplicates.

`orders_raw` uses `mode("append")` since it stores individual events.

In [ ]:
def make_mongo_writer(collection: str, id_fields: str):
    def write_batch(batch_df: DataFrame, batch_id: int) -> None:
        if batch_df.isEmpty():
            return
        row_count = batch_df.count()
        print(f"[batch {batch_id}] Writing {row_count} rows to {MONGO_DATABASE}.{collection}")
        (
            batch_df.write
            .format("mongodb")
            .mode("append")
            .option("connection.uri", MONGO_URI)
            .option("database", MONGO_DATABASE)
            .option("collection", collection)
            .option("writeConcern.w", "majority")
            .option("upsertDocument", "true")
            .option("idFieldList", id_fields)
            .save()
        )
    return write_batch

def make_raw_writer(collection: str = "orders_raw"):
    """Appends raw enriched events to MongoDB."""
    def write_batch(batch_df: DataFrame, batch_id: int) -> None:
        if batch_df.isEmpty():
            return
        row_count = batch_df.count()
        print(f"[batch {batch_id}] Writing {row_count} raw events to {MONGO_DATABASE}.{collection}")
        (
            batch_df
            .drop("processing_time")
            .write
            .format("mongodb")
            .mode("append")
            .option("connection.uri", MONGO_URI)
            .option("database", MONGO_DATABASE)
            .option("collection", collection)
            .option("writeConcern.w", "majority")

            .save()
        )
    return write_batch

print("MongoDB writer functions defined.")

## 10. Start Streaming Queries

> Interrupt the kernel to stop all queries.

In [ ]:
q1 = (
    cat_stream.writeStream
    .outputMode("complete")
    .option("checkpointLocation", f"{CHECKPOINT_DIR}/revenue_by_category")
    .foreachBatch(make_mongo_writer("revenue_by_category", "window_start,category"))
    .trigger(processingTime="30 seconds")
    .start()
)

q2 = (
    country_stream.writeStream
    .outputMode("complete")
    .option("checkpointLocation", f"{CHECKPOINT_DIR}/orders_by_country")
    .foreachBatch(make_mongo_writer("orders_by_country", "window_start,shipping_country"))
    .trigger(processingTime="30 seconds")
    .start()
)

q3 = (
    payment_stream.writeStream
    .outputMode("complete")
    .option("checkpointLocation", f"{CHECKPOINT_DIR}/payment_method_stats")
    .foreachBatch(make_mongo_writer("payment_method_stats", "window_start,payment_method"))
    .trigger(processingTime="30 seconds")
    .start()
)

q4 = (
    enriched.writeStream
    .outputMode("append")
    .option("checkpointLocation", f"{CHECKPOINT_DIR}/orders_raw")
    .foreachBatch(make_raw_writer("orders_raw"))
    .trigger(processingTime="30 seconds")
    .start()
)

print("Streaming queries started:")
for q in [q1, q2, q3, q4]:
    print(f"  [{q.id}] status: {q.status['message']}")

In [ ]:
# Block until a query terminates — interrupt the kernel to stop
print("Waiting for termination… (interrupt kernel to stop)")
su.spark.streams.awaitAnyTermination()

In [ ]:
# Validation queries — run after stopping the stream

# 1. Top categories by total revenue
su.spark.read \
    .format("mongodb") \
    .option("connection.uri", MONGO_URI) \
    .option("database", MONGO_DATABASE) \
    .option("collection", "revenue_by_category") \
    .load() \
    .groupBy("category") \
    .agg(
        F.sum("total_revenue").alias("total_revenue"),
        F.sum("total_orders").alias("total_orders")
    ) \
    .orderBy(F.desc("total_revenue")) \
    .show(truncate=False)

# 2. Top countries by order count
su.spark.read \
    .format("mongodb") \
    .option("connection.uri", MONGO_URI) \
    .option("database", MONGO_DATABASE) \
    .option("collection", "orders_by_country") \
    .load() \
    .groupBy("shipping_country") \
    .agg(F.sum("order_count").alias("total_orders")) \
    .orderBy(F.desc("total_orders")) \
    .limit(5) \
    .show(truncate=False)

# 3. Payment method distribution
su.spark.read \
    .format("mongodb") \
    .option("connection.uri", MONGO_URI) \
    .option("database", MONGO_DATABASE) \
    .option("collection", "payment_method_stats") \
    .load() \
    .groupBy("payment_method") \
    .agg(F.sum("transaction_count").alias("total_transactions")) \
    .orderBy(F.desc("total_transactions")) \
    .show(truncate=False)

# 4. Raw events count
su.spark.read \
    .format("mongodb") \
    .option("connection.uri", MONGO_URI) \
    .option("database", MONGO_DATABASE) \
    .option("collection", "orders_raw") \
    .load() \
    .count()

## 11. Stop the SparkSession

In [ ]:
su.spark.stop()